In [0]:
%sql
CREATE DATABASE IF NOT EXISTS compra_venta_auto;
USE compra_venta_auto;

In [0]:
%sql
USE compra_venta_auto;

In [0]:
%sql
CREATE TABLE MARCAS(
  ID_MARCA  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  NOMBRE_MARCA VARCHAR(200),
  PAIS_ORIGEN VARCHAR(200),
  ANIO_FUNDACION INT
);

In [0]:
%sql
CREATE TABLE VEHICULOS(
  ID_VEHICULO BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  ID_MARCA BIGINT,
  NOMBRE_MODELO VARCHAR(200),
  ANIO INT,
  TIPO_COMBUSTIBLE VARCHAR(200),
  PRECIO DECIMAL (10,2),
  KILOMETRAJE INT,
  FOREIGN KEY (ID_MARCA) REFERENCES MARCAS(ID_MARCA)
);

In [0]:
%sql
CREATE TABLE CLIENTES(
  ID_CLIENTE BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  NOMBRE VARCHAR(200),
  APELLIDO VARCHAR(200),
  EMAIL VARCHAR(200),
  TELEFONO INT
);

In [0]:
%sql
CREATE TABLE PAGOS_DETALLE(
  ID_PAGO_DETALLE BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  ESTADO_PAGO VARCHAR(200) --PENDIENTE/COMPLETADO/CANCELADA
  );

In [0]:
%sql
CREATE TABLE ORDENES(
  ID_ORDEN  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  ID_CLIENTE BIGINT,
  ID_VEHICULO BIGINT,
  ID_PAGO_DETALLE BIGINT,
  TIPO_ORDEN VARCHAR(200), --COMPRA/RESERVA
  FOREIGN KEY (ID_CLIENTE) REFERENCES CLIENTES(ID_CLIENTE),
  FOREIGN KEY (ID_VEHICULO) REFERENCES VEHICULOS(ID_VEHICULO),
  FOREIGN KEY (ID_PAGO_DETALLE) REFERENCES PAGOS_DETALLE(ID_PAGO_DETALLE)
);

In [0]:
%sql
CREATE TABLE PAGOS(
  ID_PAGO  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  ID_ORDEN BIGINT,
  ID_PAGO_DETALLE BIGINT,
  MONTO DECIMAL (10,2),
  FECHA_PAGO DATE,
  METODO_PAGO VARCHAR(200),
  FOREIGN KEY (ID_ORDEN) REFERENCES ORDENES(ID_ORDEN),
  FOREIGN KEY (ID_PAGO_DETALLE) REFERENCES PAGOS_DETALLE(ID_PAGO_DETALLE)
);

In [0]:
%sql
DROP TABLE IF EXISTS MARCAS;
DROP TABLE IF EXISTS VEHICULOS;
DROP TABLE IF EXISTS CLIENTES;
DROP TABLE IF EXISTS PAGOS_DETALLE;
DROP TABLE IF EXISTS ORDENES;
DROP TABLE IF EXISTS PAGOS;
    

In [0]:
%sql
INSERT INTO MARCAS (NOMBRE_MARCA, PAIS_ORIGEN, ANIO_FUNDACION) VALUES
('BMW',     'Alemania', 1916),
('Toyota',  'Japón',    1937),
('Ford',    'EEUU',     1903),
('Audi',    'Alemania', 1909),
('Renault', 'Francia',  1899);

In [0]:
%sql
SELECT * FROM marcas;

In [0]:
%sql
INSERT INTO VEHICULOS (ID_MARCA, NOMBRE_MODELO, ANIO, TIPO_COMBUSTIBLE, PRECIO, KILOMETRAJE) VALUES
(1, 'Serie 3', 2022, 'Nafta',     35000.00, 15000),
(1, 'Serie 5', 2021, 'Diesel',    45000.00, 30000),
(2, 'Corolla', 2023, 'Híbrido',   28000.00,  5000),
(3, 'Mustang', 2020, 'Nafta',     42000.00, 40000),
(4, 'A4',      2022, 'Diesel',    38000.00, 20000),
(5, 'Megane',  2021, 'Eléctrico', 25000.00, 10000);

In [0]:
%sql
SELECT * FROM vehiculos;

In [0]:
%sql
INSERT INTO CLIENTES (NOMBRE, APELLIDO, EMAIL, TELEFONO) VALUES
('Carlos', 'García',   'carlos@email.com', 26110001),
('Laura',  'Martínez', 'laura@email.com',  26122002),
('Pedro',  'Sánchez',  'pedro@email.com',  63300003),
('Ana',    'López',    'ana@email.com',    64400004);

In [0]:
%sql
INSERT INTO PAGOS_DETALLE (ESTADO_PAGO) VALUES
('COMPLETADO'),   -- ID 1 → Serie 5 pagado
('PENDIENTE'),    -- ID 2 → Mustang pendiente
('COMPLETADO'),   -- ID 3 → Corolla pagado
('PENDIENTE');    -- ID 4 → Serie 3 pendiente

In [0]:
%sql
INSERT INTO PAGOS_DETALLE (ESTADO_PAGO) VALUES
('COMPLETADO'),
('PENDIENTE'),
('CANCELADO');

In [0]:
%sql
SELECT * FROM pagos_detalle;

In [0]:
%sql
INSERT INTO ORDENES (ID_CLIENTE, ID_VEHICULO, ID_PAGO_DETALLE, TIPO_ORDEN) VALUES
(1, 2, 1, 'COMPRA'),   
(2, 4, 2, 'RESERVA'),  
(3, 3, 1, 'COMPRA'),  
(4, 1, 2, 'COMPRA'),   
(1, 6, 3, 'RESERVA');  

In [0]:
%sql
INSERT INTO PAGOS (ID_ORDEN, ID_PAGO_DETALLE, MONTO, FECHA_PAGO, METODO_PAGO) VALUES
(1, 1, 45000.20, '2025-01-16', 'Transferencia'),
(2, 2,  5000.52, '2025-02-11', 'Contado'),
(3, 1, 28000.20, '2025-03-05', 'Financiación'),
(4, 2, 10000.10, '2025-04-01', 'Transferencia');

In [0]:
%sql
select 
VEH.NOMBRE_MODELO,
MAR.NOMBRE_MARCA
from VEHICULOS VEH 
JOIN marcas MAR ON (VEH.ID_MARCA= MAR.ID_MARCA)
;

In [0]:
%sql
SELECT 
METODO_PAGO,
SUM(MONTO) TOTAL_RECAUDADO
FROM pagos GROUP BY METODO_PAGO;


In [0]:
%sql
---SELECT * FROM clientes; 
--SELECT * FROM ordenes; CON ORDENES COMPLETADAS, CON DETALLES DEL AUTO
---SELECT * FROM pagos_detalle; 
SELECT 
CTE.APELLIDO ||" "|| CTE.NOMBRE NOMBRE_COMPLETO,
CTE.TELEFONO, 
CTE.EMAIL,
VEH.NOMBRE_MODELO,
VEH.ANIO,
VEH.TIPO_COMBUSTIBLE,
VEH.PRECIO,
VEH.KILOMETRAJE,
PAG.ESTADO_PAGO
FROM CLIENTES CTE
JOIN ORDENES ORD ON (ORD.ID_CLIENTE= CTE.ID_CLIENTE )
JOIN VEHICULOS VEH ON (VEH.ID_VEHICULO=ORD.ID_VEHICULO )
JOIN PAGOS_DETALLE PAG ON (ORD.ID_PAGO_DETALLE=PAG.ID_PAGO_DETALLE)
WHERE PAG.ESTADO_PAGO='COMPLETADO'

In [0]:
%sql
--SELECT * FROM MARCAS; RANKING DE MARCAS POR VENTAS + VENDIDA
---SELECT * FROM ORDENES;

SELECT 
VEH.NOMBRE_MODELO,
MAR.NOMBRE_MARCA,
---COUNT(ORD.ID_ORDEN),
ORD.TIPO_ORDEN
FROM VEHICULOS VEH 
JOIN MARCAS MAR ON (VEH.ID_MARCA=MAR.ID_MARCA)
JOIN ORDENES ORD ON (ORD.ID_VEHICULO=VEH.ID_VEHICULO)
WHERE ORD.TIPO_ORDEN='COMPRA'


In [0]:
%sql
SELECT 
    M.NOMBRE_MARCA,
    COUNT(O.ID_ORDEN)    AS TOTAL_ORDENES,
    SUM(P.MONTO)         AS TOTAL_FACTURADO,
    RANK() OVER (
        ORDER BY SUM(P.MONTO) DESC
    )                    AS RANKING
FROM MARCAS    M
JOIN VEHICULOS V   ON M.ID_MARCA          = V.ID_MARCA
JOIN ORDENES   O   ON V.ID_VEHICULO       = O.ID_VEHICULO
JOIN PAGOS     P   ON O.ID_ORDEN          = P.ID_ORDEN
JOIN PAGOS_DETALLE PD ON O.ID_PAGO_DETALLE = PD.ID_PAGO_DETALLE
WHERE PD.ESTADO_PAGO != 'CANCELADO'
GROUP BY M.NOMBRE_MARCA
ORDER BY RANKING;

In [0]:
%sql
SELECT * FROM ordenes;
SELECT 
    M.NOMBRE_MARCA,
    V.NOMBRE_MODELO,
    V.ANIO,
    V.TIPO_COMBUSTIBLE,
    V.PRECIO
FROM VEHICULOS V
JOIN MARCAS M ON V.ID_MARCA = M.ID_MARCA
WHERE V.ID_VEHICULO NOT IN (
    SELECT DISTINCT ID_VEHICULO 
    FROM ORDENES
)
ORDER BY M.NOMBRE_MARCA;



# Documentación — Modelo Compra Venta de auto

---

## Descripción
Modelo de datos para una plataforma de compraventa de autos usados.
Base de datos: compra_venta_auto

---

## Tablas

| Tabla | Descripción |
|-------|-------------|
| MARCAS | Almacena las marcas de vehículos disponibles |
| VEHICULOS | Catálogo de autos con estado, precio y características |
| CLIENTES | Datos personales de los clientes registrados |
| PAGOS_DETALLE | Estados posibles de pago: COMPLETADO, PENDIENTE, CANCELADO |
| ORDENES | Registro de operaciones de compra y reserva |
| PAGOS | Detalle de pagos asociados a cada orden |

---

## Relaciones

- Un *cliente* puede tener muchas *órdenes*
- Una *orden* corresponde a un solo *vehículo*
- Un *vehículo* pertenece a una sola *marca*
- Una *orden* tiene un *estado de pago* definido en PAGOS_DETALLE
- Un *pago* referencia una *orden* y un *estado de pago*

---

## Notas técnicas

> Las claves foráneas en Databricks son *informativas*.
> La integridad referencial debe validarse en el pipeline de datos.

---

## Autor

*Nombre:* Del Pino Karen
*Fecha:* 03/05/2026